# Standing-wave tube: pressure-to-velocity reconstruction

The 385 Hz measurement uses 12 valid sensor positions, checkpoint 1dtube_v31 and 100 reverse steps. MASK_SECOND_CHANNEL=True observes pressure and reconstructs velocity. Setting it to False gives paired pressure–velocity observations. Each channel is normalized using all valid positions and the full measurement record; the statistics are saved with the outputs.

Run cells in order. Sampling counts are reverse-diffusion steps; checkpoint numbers identify training state. Data and checkpoints are installed under the repository root.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'models').is_dir() and (p / 'sampler').is_dir() and (p / 'trainer').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the MRSM repository.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
OUTPUT = ROOT / 'outputs' / 'tube_pressure_velocity'
OUTPUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family': 'Arial', 'font.size': 10})

def format_current_figure():
    def tick(value, pos):
        return '0' if abs(value) < 1e-12 else f'{value:g}'
    for ax in plt.gcf().axes:
        if ax.get_xscale() == 'linear':
            ax.xaxis.set_major_formatter(FuncFormatter(tick))
        if ax.get_yscale() == 'linear':
            ax.yaxis.set_major_formatter(FuncFormatter(tick))

print('Repository:', ROOT)
print('Device:', device)


In [ ]:
import os
from sampler.utils import *
from trainer.utils import restore_checkpoint
import torch
import torch.nn as nn
import json
from collections import namedtuple
from tqdm import tqdm
from models import get_model, get_ema
from sampler import VESDE
import numpy as np
from einops import rearrange
data_name = '1dtube'
version = 'v31'
opt_path = './results/' + data_name + '_' + version + '/config.json'
with open(opt_path, 'r', encoding='utf-8') as f:
    opt = json.load(f)
    opt['num_channels'] = opt['num_components'] + opt['num_conditions']
opt['results_path'] = str(Path(opt_path).parent)
opt['device'] = device
OPT_class = namedtuple('OPT_class', opt.keys())
opt = OPT_class(**opt)
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'


In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
net = get_model(opt)
net = nn.DataParallel(net, device_ids=[0] if torch.cuda.is_available() else [])
net.to(device)
ema = get_ema(net.parameters(), decay=opt.ema_rate)
optimizer = torch.optim.Adam(net.parameters(), lr=opt.lr, betas=(0.9, 0.999), eps=1e-08, weight_decay=0)
state = dict(optimizer=optimizer, model=net, ema=ema, epoch=0, iteration=0, loss_train=[])
ckpt_path = opt.results_path + '/checkpoint.pth'
if not Path(ckpt_path).is_file():
    raise FileNotFoundError(ckpt_path)
state = restore_checkpoint(ckpt_path, state, device)
net = state['model']
ema = state['ema']
ema.copy_to(net.parameters())
print('Model loaded from epoch', state['epoch'])


In [ ]:
data_test_path = 'data/pu385.npy'
print(f'Loading data from: {data_test_path}')
data_raw = np.load(data_test_path)
print('Calculating channel normalization statistics...')
valid_sensor_indices = [1, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14]
data_for_stats = data_raw[:, :, :, valid_sensor_indices]
mean_arr = np.mean(data_for_stats, axis=(0, 1, 3))
std_arr = np.std(data_for_stats, axis=(0, 1, 3))
print(f'Channel means: {mean_arr}')
print(f'Channel standard deviations: {std_arr}')

def scalar_numpy(x):
    m = mean_arr[:, None]
    s = std_arr[:, None]
    return (x - m) / s

def scalar_inv_numpy(x):
    m = mean_arr[:, None]
    s = std_arr[:, None]
    return x * s + m

class DataScalarTorch:

    def __init__(self, mean, std, device):
        self.mean = torch.from_numpy(mean).float().to(device).view(1, 1, -1, 1)
        self.std = torch.from_numpy(std).float().to(device).view(1, 1, -1, 1)

    def __call__(self, x):
        return x * self.std + self.mean
data_scalar_torch = DataScalarTorch(mean_arr, std_arr, device)
x_test = data_raw[:, :, :, valid_sensor_indices]
m = 2
T = opt.num_frames
T_prime_target = 200
B = int(np.ceil((T_prime_target - m) / (T - m)))
T_prime = B * (T - m) + m
start_t = 100
single_sample_data = x_test[:, start_t:start_t + T_prime, :, :]
samples = scalar_numpy(single_sample_data).astype(np.float32)
print(f'Samples Shape: {samples.shape}')
print('-' * 40)
print('Normalized-window statistics:')
for ch in [0, 1]:
    d = samples[:, :, ch, :]
    print(f'Ch {ch}: MeanAbs={np.mean(np.abs(d)):.4f}, Var={np.var(d):.4f}, Min={np.min(d):.4f}, Max={np.max(d):.4f}')
print('-' * 40)
if np.mean(np.abs(samples[:, :, 1, :])) > 10:
    print('Warning: mean absolute normalized velocity exceeds 10; check the input data and scaling.')
else:
    print('Velocity-magnitude threshold: 10.')


In [ ]:
def scalar_numpy(x):
    m = mean_arr[:, None]
    s = std_arr[:, None]
    return (x - m) / s

def scalar_inv_numpy(x):
    m = mean_arr[:, None]
    s = std_arr[:, None]
    return x * s + m
data_scalar_torch = DataScalarTorch(mean_arr[:opt.num_components], std_arr[:opt.num_components], device)


In [ ]:
import torch
import matplotlib.pyplot as plt
MASK_SECOND_CHANNEL = True
print(f'==========================================')
print(f"Observation mode: {('pressure only' if MASK_SECOND_CHANNEL else 'pressure and velocity')}")
print(f'==========================================')
outer_loop = 100
denoising_steps = 100
denoising_steps2 = 100
cof = 0.01
seed_sample = 123
v_channel_idx = 1
velocity_mean = mean_arr[v_channel_idx]
velocity_std = std_arr[v_channel_idx]
norm_v_zero = (0.0 - velocity_mean) / velocity_std
alpha_case = 0.5
beta = 0.4
gamma = 0.5
physics_params = {'physics_c': 343.0, 'physics_rho0': 1.21, 'physics_dt': 0.0001, 'physics_dx': 0.02, 'reg_coef_wave': 10.0 * cof, 'reg_coef_cont': 10.0 * cof, 'reg_coef_euler': 10.0 * cof, 'reg_coef_bc': 0 * cof, 'normalized_velocity_zero': None}
full_sensor_list = np.array([1, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14])
sensors_to_hide = None
sensors_to_keep = np.setdiff1d(full_sensor_list, sensors_to_hide)
print(f'Observed sensor positions: {sensors_to_keep}')
kept_indices_np = sensors_to_keep * 4
kept_indices = torch.from_numpy(kept_indices_np).long().to(device)
spatial_sparsity = 1 - len(sensors_to_keep) / 64.0
if MASK_SECOND_CHANNEL:
    sparsity = (spatial_sparsity + 1.0) / 2.0
else:
    sparsity = spatial_sparsity
alpha = alpha_case / np.sqrt(1 - sparsity)
print(f'Spatial Sparsity: {spatial_sparsity:.4f}')
print(f'Final Sparsity used: {sparsity:.4f}')
print(f'Calculated alpha: {alpha:.4f}')
samples_scale = samples.copy()
keep_mask = np.isin(full_sensor_list, sensors_to_keep)
y = samples_scale[..., keep_mask]
if MASK_SECOND_CHANNEL:
    print('Masking velocity observations...')
    y[..., 1, :] = 0.0
else:
    print('Using pressure and velocity observations...')
mask_vals = [1.0, 0.0] if MASK_SECOND_CHANNEL else [1.0, 1.0]
channel_mask_tensor = torch.tensor(mask_vals, device=device).view(1, 1, 2, 1)

def transform(x):
    out = x[:, :, :opt.num_components, kept_indices]
    return out * channel_mask_tensor
transform_gen = transform
sde = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=outer_loop)
sde1 = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=denoising_steps)
sde2 = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=denoising_steps2)
net.eval()
reap_fn = lambda x, num: torch.stack([x for _ in range(num)])
T_chunk = opt.num_frames
latent_mask = torch.ones([T_chunk, 1, 1]).float().to(device)
obs_mask = torch.zeros([T_chunk, 1, 1]).float().to(device)
frame_indices = torch.arange(T_chunk).to(device)
net_fn = lambda x, t: net(x, x0=x, timesteps=t, latent_mask=reap_fn(latent_mask, len(x)), obs_mask=reap_fn(obs_mask, len(x)), frame_indices=reap_fn(frame_indices, len(x)))[0]
setup_seed(seed_sample)
print('Sampling SDE...')
x_inpainted = s3gm_sample_1d(opt, net_fn, sde, y, transform_gen, alpha=alpha, beta=beta, gamma=gamma, T_prime_y=T_prime, T_prime=T_prime, overlap=m, corrector=NoneCorrector, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True)
setup_seed(seed_sample)
print('Sampling ISS...')
x_inpainted1 = s3gm_sample_1d_ISS(opt, net_fn, sde1, y, transform_gen, alpha=alpha, beta=beta, gamma=gamma, T_prime_y=T_prime, T_prime=T_prime, overlap=m, denoising_steps=denoising_steps, eta=0, corrector=NoneCorrector, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True, reg_coef_pa=0)
setup_seed(seed_sample)
print('Sampling PI-ISS...')
x_inpainted2 = s3gm_sample_1d_ISS_1dtube_res(opt, net_fn, sde2, y, transform_gen, alpha=0.5 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime, T_prime=T_prime, overlap=m, denoising_steps=denoising_steps2, eta=0, reg_coef_pa=0.1 * T_prime, data_scalar=data_scalar_torch, physics_start_step=denoising_steps2, **physics_params, corrector=NoneCorrector, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True)

def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return x
x_pred = to_numpy(x_inpainted).copy()
x_pred1 = to_numpy(x_inpainted1).copy()
x_pred2 = to_numpy(x_inpainted2).copy()
x_pred[:, :, :opt.num_components] = scalar_inv_numpy(x_pred[:, :, :opt.num_components])
x_pred1[:, :, :opt.num_components] = scalar_inv_numpy(x_pred1[:, :, :opt.num_components])
x_pred2[:, :, :opt.num_components] = scalar_inv_numpy(x_pred2[:, :, :opt.num_components])
print(f'Reconstructed field shape: {x_pred.shape}')
torch.cuda.empty_cache()


## Pressure–velocity evaluation


In [ ]:
samples_raw = scalar_inv_numpy(samples)
positions = np.asarray(valid_sensor_indices) * 4
rows = []
(fig, axes) = plt.subplots(1, 2, figsize=(10, 4))
for (method, pred) in [('SDE', x_pred), ('ISS', x_pred1), ('PI-ISS', x_pred2)]:
    for (ch, name) in enumerate(['Pressure', 'Velocity']):
        truth = samples_raw[0, :, ch, :]
        estimate = pred[0, :, ch, :][:, positions]
        error = cal_rmse(truth[:, None, :], estimate[:, None, :], normalize=True)
        rows.append({'method': method, 'channel': name, 'nrmse': float(np.mean(error))})
        axes[ch].plot(np.ravel(error), label=method)
for (ax, name) in zip(axes, ['Pressure', 'Velocity']):
    ax.set(xlabel='Physical time frame', ylabel='nRMSE', title=name)
    ax.legend()
format_current_figure()
fig.savefig(OUTPUT / 'pressure_velocity_errors.pdf', bbox_inches='tight')
format_current_figure()
plt.show()
pd.DataFrame(rows).to_csv(OUTPUT / 'pressure_velocity_metrics.csv', index=False)
pd.DataFrame({'channel': ['pressure', 'velocity'], 'mean': mean_arr, 'std': std_arr}).to_csv(OUTPUT / 'normalization.csv', index=False)
